# Experiment 10 — composite scores for ranking and portfolio allocation

## Goal and method

Expand the original [NB10 snapshot study](appendices/10-original-portfolio-snapshots.ipynb): inverse variance **and other metrics are ingredients inside per-vault scores**, with separate recipes for ranking and weighting. Build on the existing scoring variants and sparse-data feature definitions from experiments 04–09. This implements the [expanded reviewed plan](10-expanded-score-allocation-plan.md).

For example, `Q = (F + I90 + K90) / 3` is calculated for the full eligible universe before selecting a basket. Only the final Q is mapped into requested position weights. We compare arithmetic, geometric and minimum scores; standalone CAGR/Sharpe/Sortino/drawdown/Calmar/K; score sizing maps; score risk budgets; missing-data policies; six versus ten positions; and entry constraints.

The **smoke study runs 36 combinations over the full available 2026 history**, on a four-day decision cadence from 1 January to 4 October. It retains all **70 origins: 62 mature 30-day outcome windows, eight pending**. All available **2025 indicator warmup** and earlier recorded observations are retained for longer windows and maximum-history K. Sparse NAV is carried causally; observation counts and permission/capacity clocks remain separate. Young vaults remain eligible when long-window metrics are unavailable.

Each basket holds fixed units for 30 days. Overlapping snapshots give descriptive allocation comparisons, not executable strategy CAGR. Current universe/guard and publication-clock limitations remain. The ten-complete-block inference gate is unmet in 2026. The broader all-139 recipe and multi-horizon stages are proposed next work, not part of this executed smoke scope.

## Key new insights

On 62 common mature origins, adding the inverse-variance component to the focal weighting score improves mean 30-day return by **0.165 percentage points** versus focal-score sizing, and **0.145 points** versus a neutral-offset control. Including it in ranking instead reduces return by **1.112 points** versus the focal ranking/equal-sizing control. Keeping ranking fixed, the tested K weighting addition changes return by **−0.014 points**. These are specific recipe comparisons, not general metric verdicts.

## Summary of results

All **36 policies × 70 scheduled origins** completed. Exact source and focal equal-weight controls reproduce the original saved ledgers. There are **27 passing audit checks** and **15 separate inspected charts**, with matched budgets, pending labels, missingness, cap distortion and quarter/vault attribution reported. The proposed next-stage registry contains **334 unique standalone metric/lookback policies**, alongside the preserved inventory of 139 frozen scoring recipes; these broader stages have not been run.

## Robustness of results

The study retains 2025 warmup and earlier observations, forward-fills sparse NAV causally, and separates missing metrics from valid zeros. Independent arithmetic, source parity, historical NAV reconstruction and allocation checks pass. The calendar lacks the required ten complete blocks, so no statistical superiority or strategy promotion is claimed. Profit concentration, young-vault support and cash deployment need further matched checks. Snapshot returns are not strategy CAGR.

See [smoke findings](10-expanded-score-allocation-smoke-findings.md) for all matched effects, limitations and the exact next-stage scope.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'portfolio_score_design.py').exists():
    ROOT = ROOT / 'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0, str(ROOT))
from portfolio_score_design import smoke_registry
from portfolio_score_components import read_inputs, build_components
from portfolio_score_pipeline import execute_smoke
from portfolio_score_audit import run_fixtures, run_audit
from portfolio_score_report import render_report, write_findings
from score_matrix_visualisation import compact_output
policies = smoke_registry()


## Independent numerical fixtures — 2026-01-01 → 2026-10-05; 30-day horizon

Before reading outcomes, check score arithmetic, missing versus valid zero, component order, sparse filling, sizing odds, zero allocations under caps and the K formula using independent calculations.


In [2]:
fixtures = run_fixtures()
display(fixtures)


,check,passed,detail
0,Independent composite arithmetic,True,0.5S+0.25I+0.25K gives 0.6 for both vaults
1,Catch separately normalised allocation blend,True,The adverse alternative disagrees with equal f...
2,Missing policies / valid zero,True,M0/MN/MA arithmetic and masks checked independ...
3,Geometric / minimum zeros,True,Missing-zero and valid-zero remain zeros witho...
4,Zero coefficient exclusion,True,Unavailable zero-coefficient inputs cannot mak...
5,Peer-relative I/V invariant,True,Same relative-risk order and scale invariance;...
6,Midrank ties and singleton,True,Independent tie formula
7,Sizing odds / power,True,Full-range softmax 4:1; flattening fixture ret...
8,Neutral missing percentile,True,Unavailable neutral point is ranked against av...
9,Zero requested names stay unfunded,True,One positive name leaves 67% cash; zero names ...


## Pinned full history and indicator warmup — 2026-01-01 → 2026-10-05; 30-day horizon

Verify the archive, metadata, 139-recipe inventory and audited feature generation. Retain all 2025 and earlier input history; decision origins start in 2026. No outcomes are used to trim the grid or eligible universe.


In [3]:
with compact_output('Pinned full-history inputs; estimated under one minute'):
    data = read_inputs()
display(pd.DataFrame([{'Scheduled origins': len(data['origins']), 'Mature 30-day origins': 62, 'Pending origins': 8, 'Verified frozen recipes': len(data['registry']), 'Recorded NAV start': str(data['daily'].date.min()), 'Generation': data['fingerprint'][:16]}]))


Pinned full-history inputs; estimated under one minute — complete

,Scheduled origins,Mature 30-day origins,Pending origins,Verified frozen recipes,Recorded NAV start,Generation
0,70,62,8,139,2023-03-01 00:00:00,0d3861e02592ca8f


## Eligible-universe component scores — 2026-01-01 → 2026-10-05; 30-day horizon

Transform metrics and estimate risk before selecting any baskets. Inverse-risk components use leave-one-out priors and genuine update counts. Missingness stays explicit; maximum-history K retains the first recorded anchor.


In [4]:
with compact_output('Causal component tensors; estimated one–two minutes'):
    tensors = build_components(data, policies)
display(tensors['feature_inventory'])


Causal component tensors; estimated one–two minutes — complete

,component,status,observations,finite,origins
0,B,loaded_verified,12702,3275,70
1,C180,loaded_verified,12702,7430,70
2,C30,loaded_verified,12702,12289,70
3,C360,loaded_verified,12702,3574,70
4,Ca180,loaded_verified,12702,7430,70
5,Ca360,loaded_verified,12702,3574,70
6,D90,loaded_verified,12702,10526,70
7,F,loaded_verified,12702,3188,70
8,I90,computed_causally,12702,12702,70
9,K30+,loaded_verified,12702,11441,70


## Full-2026 restricted-combination smoke run — 2026-01-01 → 2026-10-05; 30-day horizon

Execute all 36 arms at every scheduled origin. Allocate before joining future labels; keep immature late dates in the ledger. Entry caps redistribute only to names with positive requested capital. Outcomes retain unresolved terminal weights and common-basket equal-budget comparisons.


In [5]:
with compact_output('36 policies × 70 origins; estimated one–three minutes'):
    result = execute_smoke(data, tensors, policies)


36 policies × 70 origins; estimated one–three minutes — complete

## Original-archive and saved-reference audit — 2026-01-01 → 2026-10-05; 30-day horizon

Reconstruct first-origin metrics from the full retained NAV, perturb future data, inspect support counts and risk contributions, and compare exact per-origin T00/T01 weights and returns with the original saved NB10 controls.


In [6]:
with compact_output('Independent warmup, causality and source parity checks'):
    audit = run_audit(data, tensors, result, policies)


Independent warmup, causality and source parity checks — complete

## Smoke results and allocation mechanisms — 2026-01-01 → 2026-10-05; 30-day horizon

Each table and separate figure below has its own description. Means use mature resolved support; pending Q4 remains visible. Matched effects and cash diagnose the mechanisms rather than promoting a strategy from researched history.


In [7]:
figures = render_report(data, tensors, result, audit, policies)


### Scheduled and mature observations — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Every decision date remains in diagnostics; immature labels cannot contribute returns. All available 2025 data and earlier history initialise indicators.

quarter,scheduled,mature,pending
2026 Q1,23,23,0
2026 Q2,23,23,0
2026 Q3,23,16,7
2026 Q4,1,0,1


### Executed design T00–T11 — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Ingredients are combined before selection; weighting maps convert one final score into requested capital. C applies entry caps; P is uncapped.

arm,ranking,weighting,map,N,missing,constraints,purpose
T00,L,source90,source,6,Legacy,C,Exact source control
T01,F,constant 1,equal,6,M0,C,Same-basket equal
T02,F,F,linear,6,M0,C,Frozen score sizing
T03,F,I90,linear,6,M0,C,Inverse variance component
T04,F,V90,linear,6,M0,C,Inverse volatility component
T05,F,"A(F,I90)",linear,6,M0,C,Risk in sizing score
T06,"A(F,I90)",constant 1,equal,6,M0,C,Risk in ranking only
T07,"A(F,I90)","A(F,I90)",linear,6,M0,C,Risk in both roles
T08,F,"A(F,V90)",linear,6,M0,C,Inverse volatility blend
T09,F,"G(F,I90)",linear,6,M0,C,Geometric agreement


### Executed design T12–T23 — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Ingredients are combined before selection; weighting maps convert one final score into requested capital. C applies entry caps; P is uncapped.

...

### Tail and carried-mark diagnostics — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Predeclared source/focal controls and standalone probes show mean versus median, worst/best origin and first-update contributions. A concentrated tail is not proof of a data error.

arm,mean_return,median_return,worst_origin,best_origin,median_cash,first_update,remaining,carried_anchor_weight,unresolved_weight
T00,1.47%,1.48%,-12.56%,15.64%,2.00%,0.06%,1.41%,10.52%,0.00%
T01,1.72%,1.12%,-14.61%,25.13%,2.00%,0.09%,1.63%,10.41%,0.00%
T02,1.65%,1.33%,-12.55%,25.13%,2.00%,0.08%,1.57%,10.30%,0.00%
T05,1.81%,1.20%,-9.60%,25.13%,2.00%,0.08%,1.74%,10.32%,0.00%
T14,1.58%,0.94%,-9.70%,22.29%,2.00%,0.18%,1.40%,11.54%,0.00%
T18,2.39%,1.27%,-8.89%,18.86%,2.00%,0.09%,2.30%,10.28%,0.00%
T19,-2.86%,-1.45%,-25.36%,10.37%,5.67%,-0.03%,-2.83%,9.05%,0.00%
T27,1.67%,1.12%,-12.21%,25.13%,2.00%,0.08%,1.59%,10.34%,0.00%
T28,-3.31%,-2.74%,-39.06%,33.64%,19.04%,-0.64%,-2.67%,8.17%,1.27%
T29,2.41%,2.40%,-14.42%,20.30%,2.00%,-0.05%,2.46%,11.00%,0.07%


### Largest vault contributions — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Three largest absolute contributions for predeclared control/blend/short-CAGR arms. Contributions average across the arm’s resolved origins and sum to mean snapshot return; baskets remain fixed for attribution.

arm,vault_name,appearances,mean_return_contribution
T00,Realist Capital,40,1.08%
T00,Mahamor,7,-0.57%
T00,pmalt,34,0.52%
T05,Hyperrr,22,1.21%
T05,Realist Capital,40,1.02%
T05,pmalt,33,0.34%
T14,Hyperrr,20,1.01%
T14,Realist Capital,21,0.49%
T14,pmalt,30,0.29%
T28,Not In Employment Education or Training,12,2.42%


### Sensitivity to excluding an origin quarter — 2026-01-01 → 2026-10-05 (end exclusive, UTC); 30-day horizon

Full 2026 grid / restricted combinations. Difference in mean 30-day return after removing one quarter, holding every original basket fixed. Q4 has no mature labels and is therefore shown as unavailable. This diagnoses regime concentration; it does not select a favourable period.

## Findings and next-stage inventory — 2026-01-01 → 2026-10-05; 30-day horizon

Write the smoke conclusions and the exact proposed standalone metric/lookback policy registry. Broad stages remain unexecuted; the smoke stage ends after audit, rendering and findings.


In [8]:
findings = write_findings(data, result, audit)
next_stage = __import__('json').loads((data['artifact'] / 'next-stage-registry.json').read_text())
display(pd.DataFrame([{'Audits passed': int(audit.passed.sum()), 'Smoke policies': 36, 'Scheduled origins': 70, 'Separate figures': len(figures), 'Proposed A0 policies (not run)': next_stage['unique_policies'], 'Execution seconds': data['execution_seconds']}]))
display(Markdown('[Detailed smoke findings](10-expanded-score-allocation-smoke-findings.md)'))


,Audits passed,Smoke policies,Scheduled origins,Separate figures,Proposed A0 policies (not run),Execution seconds
0,27,36,70,15,334,25.678497


[Detailed smoke findings](10-expanded-score-allocation-smoke-findings.md)